# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Seif-2/ML-week-1-FLY/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

Final queue: rank the model-scored test-split pages (the honest, grouped-split Random Forest from w05/w06) by predicted probability, attach a plain-word reason code and a concrete action. This is what an editor actually opens Monday morning.


In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
visible = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0)].copy()
tier_median = visible.groupby("position_tier")["ctr"].transform("median")
visible["tier_median_ctr"] = tier_median
visible["ctr_gap"] = tier_median - visible["ctr"]
visible["underperform_flag"] = (visible["ctr_gap"] > 0).astype(int)

num_feats = ["impressions_90d", "avg_position", "content_age_days", "days_since_last_update", "word_count"]
cat_feats = ["content_type", "main_intent", "position_tier"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
tr_idx, te_idx = next(gss.split(visible, groups=visible["client_id"]))
train, test = visible.iloc[tr_idx].copy(), visible.iloc[te_idx].copy()

Xtr = pd.get_dummies(train[num_feats + cat_feats], columns=cat_feats).fillna(0)
Xte = pd.get_dummies(test[num_feats + cat_feats], columns=cat_feats).fillna(0)
Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)

rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1).fit(Xtr, train["underperform_flag"])
test["model_score"] = rf.predict_proba(Xte)[:, 1]

# Plain-word reason codes, derived from the same honest features the model used
def reason_code(row):
    if row["avg_position"] > 10:
        return "weak_position_high_traffic"
    if row["word_count"] < 800:
        return "thin_content_visible"
    return "model_flagged_pattern"

test["reason_code"] = test.apply(reason_code, axis=1)
test["action"] = "review_title_meta_snippet"

queue = test.sort_values("model_score", ascending=False).reset_index(drop=True)
output_cols = ["content_id", "client_id", "position_tier", "avg_position", "impressions_90d",
               "word_count", "model_score", "reason_code", "action"]
queue_out = queue[output_cols]

print(f"Final ranked queue: {len(queue_out)} pages (test-split only, honest grouped split)")
print(f"Precision@50 on this queue: {queue['underperform_flag'].head(50).mean():.3f}")
print(f"Precision@200 on this queue: {queue['underperform_flag'].head(200).mean():.3f}")
queue_out.head(10)


Final ranked queue: 1461 pages (test-split only, honest grouped split)
Precision@50 on this queue: 0.520
Precision@200 on this queue: 0.550


,content_id,client_id,position_tier,avg_position,impressions_90d,word_count,model_score,reason_code,action
0,content_7b54202bac86,client_e629fa6598,page_3_5,48.5,2696,NaN,0.755475,weak_position_high_traffic,review_title_meta_snippet
1,content_0f1399caa1ae,client_e629fa6598,page_3_5,49.0,1123,NaN,0.728650,weak_position_high_traffic,review_title_meta_snippet
2,content_795234993c79,client_e629fa6598,page_3_5,46.3,1199,NaN,0.726405,weak_position_high_traffic,review_title_meta_snippet
3,content_b72b06434946,client_e629fa6598,page_3_5,41.0,1061,NaN,0.722111,weak_position_high_traffic,review_title_meta_snippet
4,content_c727f5bd7c1a,client_e629fa6598,page_3_5,41.3,908,NaN,0.720632,weak_position_high_traffic,review_title_meta_snippet
5,content_df23b4bc766d,client_e629fa6598,page_3_5,49.6,665,NaN,0.719551,weak_position_high_traffic,review_title_meta_snippet
6,content_57a091adbcdf,client_e629fa6598,page_3_5,44.5,1097,NaN,0.718126,weak_position_high_traffic,review_title_meta_snippet
7,content_f2d9832b9e02,client_4ec9599fc2,page_3_5,44.1,782,NaN,0.716845,weak_position_high_traffic,review_title_meta_snippet
8,content_dc01e2cc035d,client_e629fa6598,page_3_5,47.1,529,NaN,0.715364,weak_position_high_traffic,review_title_meta_snippet
9,content_165eddf73584,client_e629fa6598,page_3_5,44.4,1105,NaN,0.714863,weak_position_high_traffic,review_title_meta_snippet


## 2. Intended use and limits

**Who uses this:** a FlyRank content editor or SEO strategist with limited weekly review capacity — someone who needs to pick 20–50 pages out of thousands to look at first.

**What they do with it:** open the top of the queue, check the reason code, and rewrite the title/meta/snippet for pages that genuinely look off.

**Where it stops being valid:**
- This is a same-window (90-day) comparison, not a future prediction — it says a page looks off *right now* relative to peers, not that it *will decline* or *will improve* if edited.
- Built on the 30k-row starter slice, one snapshot in time — not validated across seasons or the full warehouse.
- The model was deliberately built WITHOUT looking at CTR directly (w05's honest-feature discipline), so it's weaker than a rule that just measures CTR outright — that's a feature of the design, not a flaw, but it means the model's precision (0.52–0.55) is modest, not stunning.
- Never valid for making causal claims ("this caused a decline") or claims about Google's algorithm.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

**What a person must check before acting on any row:**
- Does the page's actual title/meta look genuinely weak, or does the gap look explained by something else (search intent mismatch, a strong competing result, seasonality)?
- Is the traffic volume (`impressions_90d`) comfortably above the 500 minimum floor, or is this pick riding on a small, noisy sample?
- Does the `reason_code` make sense on inspection, or does it look like a coincidence?

**What should never be automated:**
- Auto-publishing a rewritten title/meta without human review — this queue is a starting point for attention, not a content-generation pipeline.
- Treating a high `model_score` as proof the page is broken — it's a prioritization signal, not a diagnosis.
- Extending this queue's recommendations to any client not represented in the training data without re-validating — remember w03's finding that this slice silently excludes late-onboarding clients.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

**Signs the recommendations have gone stale:**
- The base rate (currently ~0.455 on this slice) drifts significantly — a changing base rate means the label's meaning is shifting and old thresholds may misfire.
- Precision@50/@200 on a fresh evaluation slice drops meaningfully below the ~0.52/0.55 reported here — a sign the pattern the model learned (weak position + thin content + traffic) stopped holding.
- A large batch of new clients or content types enters that weren't represented in training — per w03's limitation, this data silently drops clients who onboarded late; a growing unrepresented population is a retrain trigger.
- Position-tier CTR medians (Signal A from w04) shift noticeably — since the label itself is defined relative to those medians, a real shift in the search landscape changes what "underperforming" even means.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

Writing the final queue and a metrics receipt to `work/outputs/` — the CSV stays gitignored (regenerable), the JSON metrics file gets committed as the receipt.


In [5]:
import json

os.makedirs("work/outputs", exist_ok=True)
queue_out.to_csv("work/outputs/final_action_playbook.csv", index=False)

metrics = {
    "model": "RandomForestClassifier (n_estimators=300, max_depth=8, random_state=42)",
    "split": "GroupShuffleSplit by client_id, test_size=0.3, random_state=42",
    "n_train": int(len(train)),
    "n_test": int(len(test)),
    "base_rate_test": float(test["underperform_flag"].mean()),
    "precision_at_50": float(queue["underperform_flag"].head(50).mean()),
    "precision_at_200": float(queue["underperform_flag"].head(200).mean()),
    "baseline_precision_at_50": 1.0,
    "baseline_precision_at_200": 1.0,
    "baseline_caveat": "Baseline scores near-perfect by construction (uses ctr directly, which the label is thresholded from) — see w05 Section 1.",
    "random_split_precision_at_50_naive": 0.94,
    "grouped_split_precision_at_50_honest": 0.52,
    "memorization_gap_note": "Random split overstates Precision@50 by ~0.42 vs grouped-by-client split — see w06."
}

with open("work/outputs/w07_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Written: work/outputs/final_action_playbook.csv (gitignored, regenerable)")
print("Written: work/outputs/w07_metrics.json (committed receipt)")
print(json.dumps(metrics, indent=2))


Written: work/outputs/final_action_playbook.csv (gitignored, regenerable)
Written: work/outputs/w07_metrics.json (committed receipt)
{
  "model": "RandomForestClassifier (n_estimators=300, max_depth=8, random_state=42)",
  "split": "GroupShuffleSplit by client_id, test_size=0.3, random_state=42",
  "n_train": 15265,
  "n_test": 1461,
  "base_rate_test": 0.45516769336071183,
  "precision_at_50": 0.52,
  "precision_at_200": 0.55,
  "baseline_precision_at_50": 1.0,
  "baseline_precision_at_200": 1.0,
  "baseline_caveat": "Baseline scores near-perfect by construction (uses ctr directly, which the label is thresholded from) \u2014 see w05 Section 1.",
  "random_split_precision_at_50_naive": 0.94,
  "grouped_split_precision_at_50_honest": 0.52,
  "memorization_gap_note": "Random split overstates Precision@50 by ~0.42 vs grouped-by-client split \u2014 see w06."
}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.